In [138]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
from typing import List, Union, Optional
from numpy.lib.stride_tricks import sliding_window_view
from src.models_paper import MG_Daily

In [2]:
yf.download(tickers=['AAPL', 'TSLA'])

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  2 of 2 completed


Price            Close                    High                     Low  \
Ticker            AAPL        TSLA        AAPL        TSLA        AAPL   
Date                                                                     
1980-12-12    0.098726         NaN    0.099155         NaN    0.098726   
1980-12-15    0.093575         NaN    0.094005         NaN    0.093575   
1980-12-16    0.086707         NaN    0.087136         NaN    0.086707   
1980-12-17    0.088853         NaN    0.089282         NaN    0.088853   
1980-12-18    0.091429         NaN    0.091858         NaN    0.091429   
...                ...         ...         ...         ...         ...   
2025-03-28  217.899994  263.549988  223.809998  276.100006  217.679993   
2025-03-31  222.130005  259.160004  225.619995  260.559998  216.229996   
2025-04-01  223.190002  268.459991  223.679993  277.450012  218.899994   
2025-04-02  223.889999  282.760010  225.190002  284.989990  221.020004   
2025-04-03  203.190002  267.279999  207.490005  276.299988  201.250000   

Price                         Open                 Volume               
Ticker            TSLA        AAPL        TSLA       AAPL         TSLA  
Date                                                                    
1980-12-12         NaN    0.098726         NaN  469033600          NaN  
1980-12-15         NaN    0.094005         NaN  175884800          NaN  
1980-12-16         NaN    0.087136         NaN  105728000          NaN  
1980-12-17         NaN    0.088853         NaN   86441600          NaN  
1980-12-18         NaN    0.091429         NaN   73449600          NaN  
...                ...         ...         ...        ...          ...  
2025-03-28  260.570007  221.669998  275.579987   39818600  123809400.0  
2025-03-31  243.360001  217.009995  249.309998   65299300  134008900.0  
2025-04-01  259.250000  219.809998  263.799988   36412700  146486900.0  
2025-04-02  251.270004  221.320007  254.600006   35905900  212787800.0  
2025-04-03  261.510010  205.539993  265.290009  103204700  135752400.0  

[11168 rows x 10 columns]

In [3]:
test = np.random.randn(20).reshape(4, 5)
print(test)
test2 = sliding_window_view(test, (2, 5))
print(sliding_window_view(test, (2, 5)).shape)

[[ 0.89541255  1.67714316  1.48584796 -1.43836793  0.05633612]
 [ 0.04119988  2.75300263  1.29544697 -1.40949936 -0.04282183]
 [ 0.39820527 -1.87599911 -0.05389647  0.21720529 -1.22300458]
 [-0.24752396 -0.9413972  -0.93249942 -0.83202673 -0.53886309]]
(3, 1, 2, 5)


In [4]:
yf.download()

TypeError: download() missing 1 required positional argument: 'tickers'

In [117]:
class Yahoo_Downloader:
    
    def __init__(self, tickers:List[str], date_start: str="2010-07-01", date_end="2019-07-01", stride:int=1):
        self.tickers = tickers
        self.date_start = date_start
        self.date_end = date_end
        self.n_assets = len(tickers)
        self.stride = 1
        self.pd_data = yf.download(tickers=self.tickers, start=date_start, end=date_end)
        self.raw_data = self.pd_data.values
        self.n_samples = self.raw_data.shape[0]
        self.n_features = self.raw_data.shape[1]
        
        if self.pd_data.isnull().sum().sum() > 0:
            raise ValueError('Utilisez des actions sans valeurs manquantes ou bien choisissez une autre période')
        
    def _compute_data_training(self, N:int=5):
        
        #compute la data pour l'entrainement, doit retourner des fenetres de taille N, avec un décalage de 1 jour (stride)
        self.first_index_window = N #l'index associé à la prédiction de la première fenetre
        X = sliding_window_view(self.raw_data, (N, self.raw_data.shape[1])).reshape(-1, N, self.raw_data.shape[1])
        
        close_data = self.pd_data['Close'].values
        n_windows = self.n_samples - N + 1
        
        y = sliding_window_view( (close_data[1:] > close_data[0:-1]).astype(int), (N, self.n_assets) ).reshape(-1, N, self.n_assets)

        return X[:-1], y
    
    def compute_train_val_test(self, N:int=5, tensor:bool=False):
        
        X, y = self._compute_data_training(N)
        #on garde 1 an pour la val et 1 an pour le test
        #on recupère aussi les indices pour facilement récupérer les périodes dans le dataframe
        train_rate = 0.8
        test_rate, val_rate = 0.1, 0.1
        
        T = X.shape[0]
        
        self.train_periods = (0, int(np.floor(train_rate*T)))
        self.val_periods = (int(np.floor(train_rate*T)), int(np.floor(train_rate*T))+int(np.floor(val_rate*T)))
        self.test_periods = (int(np.floor(train_rate*T))+int(np.floor(val_rate*T)), int(np.floor(train_rate*T))+2*int(np.floor(val_rate*T)))
        
        Xtrain, ytrain = X[self.train_periods[0]:self.train_periods[1], :, :], y[self.train_periods[0]:self.train_periods[1]]
        Xval, yval = X[self.val_periods[0]:self.val_periods[1], :, :], y[self.val_periods[0]:self.val_periods[1]]
        Xtest, ytest = X[self.test_periods[0]:self.test_periods[1], :, :], y[self.test_periods[0]:self.test_periods[1]]
        
        if torch:
            return torch.tensor(Xtrain), torch.tensor(ytrain), torch.tensor(Xval), torch.tensor(yval), torch.tensor(Xtest), torch.tensor(ytest)
        
        return Xtrain, ytrain, Xval, yval, Xtest, ytest
    
    
        
        
    
        
        
        
        


In [ ]:
N = 5

In [141]:


y_ = Yahoo_Downloader(tickers=['AAPL'])
X, y = y_._compute_data_training(5)
Xtrain, ytrain, Xval, yval, Xtest, ytest = y_.compute_train_val_test(N=N, tensor=True)
# Xtrain = y_.compute_train_val_test(5)

[*********************100%***********************]  1 of 1 completed


In [143]:
Xtrain.shape

torch.Size([1807, 5, 5])

In [144]:
N = 5

model = MG_Daily(N=N, F=Xtrain.shape[-1], n_heads=4, n_actifs=1)

ValueError: La taille d'entrée (5) doit être divisible par le nombre de têtes (4).